In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install huggingface_hub
from huggingface_hub import login
import shutil

from huggingface_hub import snapshot_download

HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)

path = snapshot_download(
    repo_id="ibrahimhamamci/DENTEX",
    repo_type="dataset"
)

print(path)

print("DENTEX ready")

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/datasets--ibrahimhamamci--DENTEX/snapshots/7b27ccc8e342dcb774f69adc6ca5e6c09fefce93
DENTEX ready


In [2]:
import zipfile
import os

base_path = "/root/.cache/huggingface/hub/datasets--ibrahimhamamci--DENTEX/snapshots/7b27ccc8e342dcb774f69adc6ca5e6c09fefce93/DENTEX"

for file in ["training_data.zip", "validation_data.zip", "test_data.zip"]:
    zip_path = os.path.join(base_path, file)
    extract_path = os.path.join(base_path, file.replace(".zip", ""))

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    print(f"Extracted {file}")

Extracted training_data.zip
Extracted validation_data.zip
Extracted test_data.zip


In [13]:
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.3 MB/s eta 0:00:00a 0:00:01


# Yolo Disease cross Validation

In [6]:
! pip install ultralytics 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.2 MB/s eta 0:00:00a 0:00:01


# TRAIN 

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import json
import os
from pathlib import Path
from collections import Counter

import numpy as np
from sklearn.model_selection import StratifiedKFold
from ultralytics import YOLO


# =============================================================================
# DATASET PATHS
# =============================================================================
TRAIN_JSON = "/root/.cache/huggingface/hub/datasets--ibrahimhamamci--DENTEX/snapshots/7b27ccc8e342dcb774f69adc6ca5e6c09fefce93/DENTEX/training_data/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json"

TRAIN_IMGS = "/root/.cache/huggingface/hub/datasets--ibrahimhamamci--DENTEX/snapshots/7b27ccc8e342dcb774f69adc6ca5e6c09fefce93/DENTEX/training_data/training_data/quadrant-enumeration-disease/xrays"

VAL_JSON = "/root/.cache/huggingface/hub/datasets--ibrahimhamamci--DENTEX/snapshots/7b27ccc8e342dcb774f69adc6ca5e6c09fefce93/DENTEX/validation_triple.json"

VAL_IMGS = "/root/.cache/huggingface/hub/datasets--ibrahimhamamci--DENTEX/snapshots/7b27ccc8e342dcb774f69adc6ca5e6c09fefce93/DENTEX/validation_data/validation_data/quadrant_enumeration_disease/xrays"

OUTPUT_DIR = str(Path("./dentex_yolo_cv").resolve())


# =============================================================================
# CLASS NAMES
# =============================================================================
DISEASE_NAMES = {
    0: "Impacted",
    1: "Caries",
    2: "Periapical Lesion",
    3: "Deep Caries"
}


# =============================================================================
# TRAINING SETTINGS
# =============================================================================
N_FOLDS = 5
SEED = 42

MODEL_WEIGHTS = "yolo11m.pt"

EPOCHS = 60
IMG_SIZE = 1000
BATCH = 16

METRICS = [
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
]


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def load_coco(json_path):
    with open(json_path) as f:
        return json.load(f)


def dominant_disease(key):
    # FIX: key is now ("train", id) or ("val", id) — no ambiguity
    annotations = dataset[key]["annotations"]
    if not annotations:
        return -1
    classes = [ann["category_id_3"] for ann in annotations]
    return Counter(classes).most_common(1)[0][0]


def coco_to_yolo(bbox, img_width, img_height):
    x, y, w, h = bbox
    return (
        (x + w / 2) / img_width,
        (y + h / 2) / img_height,
        w / img_width,
        h / img_height
    )


def write_split(split_name, keys, base_dir):

    image_dir = Path(base_dir) / split_name / "images"
    label_dir = Path(base_dir) / split_name / "labels"

    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    for key in keys:

        sample = dataset[key]

        info        = sample["info"]
        annotations = sample["annotations"]

        filename   = info["file_name"]
        img_width  = info["width"]
        img_height = info["height"]

        src = Path(sample["source"]) / filename
        dst = image_dir / filename

        if src.exists() and not dst.exists():
            os.symlink(src, dst)

        label_path = label_dir / f"{Path(filename).stem}.txt"

        with open(label_path, "w") as f:
            for ann in annotations:
                cls = ann["category_id_3"]
                x, y, w, h = coco_to_yolo(
                    ann["bbox"],
                    img_width,
                    img_height
                )
                f.write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

    print(f"  [{split_name}] {len(keys)} images → {base_dir}")


# =============================================================================
# LOAD DATA
# =============================================================================
train_data = load_coco(TRAIN_JSON)
val_data   = load_coco(VAL_JSON)


# =============================================================================
# BUILD UNIFIED DATASET — FIXED
# FIX: key is ("train", image_id) or ("val", image_id)
#      so overlapping numeric IDs never overwrite each other
# =============================================================================
dataset = {}

# Train images
for img in train_data["images"]:
    key = ("train", img["id"])
    dataset[key] = {
        "info":        img,
        "source":      TRAIN_IMGS,
        "annotations": []
    }

# Val images — safe, different key prefix
for img in val_data["images"]:
    key = ("val", img["id"])
    dataset[key] = {
        "info":        img,
        "source":      VAL_IMGS,
        "annotations": []
    }

# Train annotations — tagged to train entries only
for ann in train_data["annotations"]:
    key = ("train", ann["image_id"])
    if key in dataset:
        dataset[key]["annotations"].append(ann)

# Val annotations — tagged to val entries only
for ann in val_data["annotations"]:
    key = ("val", ann["image_id"])
    if key in dataset:
        dataset[key]["annotations"].append(ann)


# =============================================================================
# DATASET STATISTICS
# =============================================================================
print(f"Total images     : {len(dataset)}")   # should now be 755
print(f"Total annotations: {sum(len(v['annotations']) for v in dataset.values())}")

class_to_images = {cls_id: set() for cls_id in DISEASE_NAMES}

for key, sample in dataset.items():
    for ann in sample["annotations"]:
        cls = ann["category_id_3"]
        class_to_images[cls].add(key)

print(f"\n{'Class':<22} {'Images containing it':>20}")
print("-" * 44)
for cls_id, name in DISEASE_NAMES.items():
    print(f"  {name:<20} {len(class_to_images[cls_id]):>20}")


# =============================================================================
# STRATIFIED LABELS
# =============================================================================
image_keys   = list(dataset.keys())   
strat_labels = [dominant_disease(k) for k in image_keys]

label_counts = Counter(strat_labels)

print(f"\nTotal images for CV: {len(image_keys)}")  
print(f"\n{'Class':<22} {'Count':>10}")
print("-" * 34)
for cls_id, name in DISEASE_NAMES.items():
    print(f"  {name:<20} {label_counts.get(cls_id, 0):>10}")

missing = [
    name for cls_id, name in DISEASE_NAMES.items()
    if label_counts.get(cls_id, 0) == 0
]
if missing:
    print(f"\nClasses with no samples: {missing}")
else:
    print("\nAll disease classes present in the dataset")


# =============================================================================
# CREATE FOLDS
# =============================================================================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(
    skf.split(image_keys, strat_labels), start=1
):

    fold_dir = f"{OUTPUT_DIR}/fold_{fold}"

    train_keys = [image_keys[i] for i in train_idx]
    val_keys   = [image_keys[i] for i in val_idx]

    write_split("train", train_keys, fold_dir)
    write_split("val",   val_keys,   fold_dir)

    yaml_path = f"{fold_dir}/dataset.yaml"
    with open(yaml_path, "w") as f:
        f.write(
            f"train: {fold_dir}/train/images\n"
            f"val:   {fold_dir}/val/images\n"
            f"nc:    {len(DISEASE_NAMES)}\n"
            f"names: {list(DISEASE_NAMES.values())}\n"
        )

    print(f"  [fold {fold}] YAML → {yaml_path}")

print("\nData preparation complete. Ready to train.")


# =============================================================================
# TRAIN YOLO
# =============================================================================
fold_results = []

for fold in range(1, N_FOLDS + 1):

    print(f"\n{'=' * 50}  FOLD {fold}/{N_FOLDS}  {'=' * 50}")

    model = YOLO(MODEL_WEIGHTS)

    results = model.train(
        data=f"{OUTPUT_DIR}/fold_{fold}/dataset.yaml",
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        project=f"{OUTPUT_DIR}/runs",
        name=f"fold_{fold}",
        exist_ok=True,
        seed=SEED,
        patience=20
    )

    fold_metrics = {}
    for metric in METRICS:
        fold_metrics[metric] = results.results_dict.get(metric)

    fold_results.append(fold_metrics)

    print(f"\n  Fold {fold} results:")
    for metric, value in fold_metrics.items():
        if value is not None:
            print(f"    {metric:<30} {value:.4f}")
        else:
            print(f"    {metric:<30} N/A")


# =============================================================================
# CROSS VALIDATION SUMMARY
# =============================================================================
print("\nCROSS-VALIDATION SUMMARY")

header = (
    f"{'Metric':<30} "
    + "  ".join(f"Fold{i+1}" for i in range(N_FOLDS))
    + "    Mean     Std"
)
print(header)
print("-" * 90)

for metric in METRICS:
    values = [fold_results[i][metric] for i in range(N_FOLDS)]
    vals_str = "  ".join(
        f"{v:.4f}" if v is not None else "N/A" for v in values
    )
    valid_values = [v for v in values if v is not None]
    mean_str = f"{np.mean(valid_values):.4f}" if valid_values else "N/A"
    std_str  = f"{np.std(valid_values):.4f}"  if valid_values else "N/A"
    print(f"  {metric:<28} {vals_str}   {mean_str}   {std_str}")


# =============================================================================
# BEST FOLD
# =============================================================================
map50_values = [fold_results[i]["metrics/mAP50(B)"] for i in range(N_FOLDS)]

best_fold = int(
    np.argmax([v if v is not None else 0 for v in map50_values])
) + 1

best_weights = f"{OUTPUT_DIR}/runs/fold_{best_fold}/weights/best.pt"

print(f"\nBest fold    : {best_fold}")
print(f"Best weights : {best_weights}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Total images     : 755
Total annotations: 3711

Class                  Images containing it
--------------------------------------------
  Impacted                              270
  Caries                                660
  Periapical Lesion                     123
  Deep Caries                           341

Total images for CV: 755

Class                       Count
----------------------------------
  Impacted                    125
  Caries                      503
  Periapical Lesion            18
  Deep Caries                  78

All disease classes present in the dataset
  [train] 604 images → /kaggle/working/dentex_yolo_cv/fold_1
  [val] 151 images → /kaggle/working/de

In [17]:
import os
import zipfile

def zip_folder(folder_path, output_zip_name):
    """Zips a folder and saves it to /kaggle/working"""
    with zipfile.ZipFile(output_zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                # arcname ensures the folder structure is preserved inside the zip
                arcname = os.path.relpath(file_path, start=os.path.dirname(folder_path))
                zipf.write(file_path, arcname)
    print(f"Successfully zipped {folder_path} to {output_zip_name}")

# Define paths
source_folder = '/kaggle/working/dentex_yolo_cv/runs/fold_5'
zip_filename = '/kaggle/working/dentex_cv_t_fold5.zip'

# Check if folder exists
if os.path.exists(source_folder):
    zip_folder(source_folder, zip_filename)
else:
    print("Folder not found!")

Successfully zipped /kaggle/working/dentex_yolo_cv/runs/fold_5 to /kaggle/working/dentex_cv_t_fold5.zip


In [16]:
# =============================================================================
# SAVE BEST FOLD VALIDATION DATA
# =============================================================================

# rebuild the split to get the val keys for the best fold
skf2 = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(
    skf2.split(image_keys, strat_labels), start=1
):
    if fold != best_fold:
        continue

    val_keys = [image_keys[i] for i in val_idx]

    best_val_data = []

    for key in val_keys:

        sample   = dataset[key]
        info     = sample["info"]
        src      = str(Path(sample["source"]) / info["file_name"])

        best_val_data.append({
            "image_path":  src,
            "image_info":  info,
            "annotations": sample["annotations"]
        })

    out_path = f"{OUTPUT_DIR}/best_fold{best_fold}_val.json"

    with open(out_path, "w") as f:
        json.dump(best_val_data, f, indent=2)

    print(f"\nBest fold val images : {len(best_val_data)}")
    print(f"Saved to             : {out_path}")


Best fold val images : 151
Saved to             : /kaggle/working/dentex_yolo_cv/best_fold5_val.json
